# Notebook 4 – Duplicate Data Handling

**Learning objective:** Identify exact and partial duplicates and remove records only when justified.

Use the IBM Telco Customer Churn dataset from Sprint 4 where available. If the CSV is unavailable, the notebook uses small illustrative data so the examples can still run. Replace illustrative business rules with rules confirmed by your project.

## 1. What are duplicate records?
Exact duplicates match across all selected columns. Partial duplicates share selected identifying fields but may differ in other columns. Duplicates may result from repeated imports, retries, or legitimate repeated events.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "customerID": ["C1", "C2", "C2", "C3", "C3"],
    "date": ["2026-01-01", "2026-01-02", "2026-01-02", "2026-01-03", "2026-01-04"],
    "charge": [50, 60, 60, 70, 75]
})
display(df)
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate rows after first occurrence:")
display(df[df.duplicated(keep="first")])

**Output explanation:** `duplicated()` marks later occurrences by default. `keep=False` marks every member of a duplicate group; `keep="last"` keeps the last occurrence for identification.

## 2. `duplicated()` and `drop_duplicates()`
Use `duplicated(subset=...)` to identify duplicates using selected columns. `drop_duplicates()` removes duplicate rows according to those columns and a `keep` rule.

In [ ]:
print("All duplicate customer/date pairs:")
display(df[df.duplicated(subset=["customerID", "date"], keep=False)])

exact_removed = df.drop_duplicates()
pair_removed = df.drop_duplicates(subset=["customerID", "date"], keep="last")
print("Original rows:", len(df))
print("After exact duplicate removal:", len(exact_removed))
print("After customer/date duplicate handling:", len(pair_removed))
display(pair_removed)

**Output explanation:** The two operations use different definitions of duplicate. The customer/date rule keeps the last record for each pair, which is only appropriate if business rules say the last record is authoritative.

## 3. Business rules for duplicate removal
Before deleting, ask:
- Is each row meant to represent a unique customer, transaction, or event?
- Which columns define identity?
- If values differ, which source or timestamp is authoritative?
- Could repeated rows represent legitimate repeated events?

Blindly deleting by customer ID can destroy valid records when each customer has multiple transactions or monthly snapshots.

In [ ]:
# Example: inspect conflicting records before deciding
conflicts = df[df.duplicated(subset=["customerID"], keep=False)].sort_values("customerID")
display(conflicts)

**Output explanation:** This reveals customers with multiple records so you can inspect whether they are duplicates or legitimate events. Document the rule, record counts before/after, and any exceptions.

### Review questions
1. Exact vs partial duplicates?
2. What does `keep=False` do?
3. Why can deleting by customer ID be dangerous?
4. Which business rule would you use to retain the latest record?